# Assignment 4 – Tree & Graph Visualization

**Student ID:** 2023xxx  
**Date:** 2026-05-03

---

## Datasets

### 1. Tree Dataset – Animal Kingdom Taxonomy
**Source:** Manually constructed based on standard biological classification (Linnaean taxonomy).  
**Description:** A hierarchical classification of selected animals from Kingdom down to Species level.  
- **Nodes** represent taxonomic ranks: Kingdom, Phylum, Class, Order, Family, Genus, Species.  
- **Edges** represent *is-a-child-of* relationships (e.g. *Mammalia* is a class within Phylum *Chordata*).

### 2. Graph Dataset – Les Misérables Character Co-occurrence
**Source:** D. E. Knuth, *The Stanford GraphBase* (1993). Widely used in network-analysis literature.  
**Description:** Characters from Victor Hugo's novel *Les Misérables* connected by co-appearance in the same scene.  
- **Nodes** represent characters.  
- **Edges** represent co-occurrence (appearing in the same chapter/scene); edge weight reflects frequency of co-occurrence.

---

In [ ]:

#!pip install networkx matplotlib plotly pyvis pandas numpy

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import plotly.graph_objects as go
import json, math
from collections import defaultdict


np.random.seed(42)
print('Libraries loaded.')

---
## Part 1 – Tree Visualization (Animal Kingdom Taxonomy)

In [ ]:

edges_tree = [
    # Kingdom → Phylum
    ('Animalia', 'Chordata'),
    ('Animalia', 'Arthropoda'),
    ('Animalia', 'Mollusca'),
    # Chordata → Class
    ('Chordata', 'Mammalia'),
    ('Chordata', 'Aves'),
    ('Chordata', 'Reptilia'),
    # Arthropoda → Class
    ('Arthropoda', 'Insecta'),
    ('Arthropoda', 'Arachnida'),
    # Mollusca → Class
    ('Mollusca', 'Gastropoda'),
    # Mammalia → Order
    ('Mammalia', 'Carnivora'),
    ('Mammalia', 'Primates'),
    ('Mammalia', 'Cetacea'),
    # Aves → Order
    ('Aves', 'Passeriformes'),
    ('Aves', 'Falconiformes'),
    # Reptilia → Order
    ('Reptilia', 'Squamata'),
    # Insecta → Order
    ('Insecta', 'Lepidoptera'),
    ('Insecta', 'Coleoptera'),
    # Arachnida → Order
    ('Arachnida', 'Araneae'),
    # Gastropoda → Family
    ('Gastropoda', 'Helicidae'),
    # Carnivora → Family
    ('Carnivora', 'Felidae'),
    ('Carnivora', 'Canidae'),
    # Primates → Family
    ('Primates', 'Hominidae'),
    # Cetacea → Family
    ('Cetacea', 'Delphinidae'),
    # Falconiformes → Family
    ('Falconiformes', 'Accipitridae'),
    # Passeriformes → Family
    ('Passeriformes', 'Corvidae'),
    # Squamata → Family
    ('Squamata', 'Pythonidae'),
    # Families → Genus
    ('Felidae', 'Panthera'),
    ('Canidae', 'Canis'),
    ('Hominidae', 'Homo'),
    ('Delphinidae', 'Tursiops'),
    ('Accipitridae', 'Aquila'),
    ('Corvidae', 'Corvus'),
    ('Pythonidae', 'Python'),
    ('Helicidae', 'Helix'),
    # Genus → Species
    ('Panthera', 'P. leo (Lion)'),
    ('Panthera', 'P. tigris (Tiger)'),
    ('Canis', 'C. lupus (Wolf)'),
    ('Canis', 'C. familiaris (Dog)'),
    ('Homo', 'H. sapiens (Human)'),
    ('Tursiops', 'T. truncatus (Bottlenose Dolphin)'),
    ('Aquila', 'A. chrysaetos (Golden Eagle)'),
    ('Corvus', 'C. corax (Common Raven)'),
    ('Python', 'P. reticulatus (Reticulated Python)'),
    ('Helix', 'H. pomatia (Roman Snail)'),
]

T = nx.DiGraph()
T.add_edges_from(edges_tree)

# Rank colours
rank_map = {
    'Animalia': 'Kingdom',
    'Chordata': 'Phylum', 'Arthropoda': 'Phylum', 'Mollusca': 'Phylum',
    'Mammalia': 'Class', 'Aves': 'Class', 'Reptilia': 'Class',
    'Insecta': 'Class', 'Arachnida': 'Class', 'Gastropoda': 'Class',
    'Carnivora': 'Order', 'Primates': 'Order', 'Cetacea': 'Order',
    'Passeriformes': 'Order', 'Falconiformes': 'Order',
    'Squamata': 'Order', 'Lepidoptera': 'Order', 'Coleoptera': 'Order', 'Araneae': 'Order',
    'Felidae': 'Family', 'Canidae': 'Family', 'Hominidae': 'Family',
    'Delphinidae': 'Family', 'Accipitridae': 'Family', 'Corvidae': 'Family',
    'Pythonidae': 'Family', 'Helicidae': 'Family',
    'Panthera': 'Genus', 'Canis': 'Genus', 'Homo': 'Genus',
    'Tursiops': 'Genus', 'Aquila': 'Genus', 'Corvus': 'Genus',
    'Python': 'Genus', 'Helix': 'Genus',
}
for n in T.nodes():
    if n not in rank_map:
        rank_map[n] = 'Species'

color_palette = {
    'Kingdom': '#e63946',
    'Phylum':  '#f4a261',
    'Class':   '#2a9d8f',
    'Order':   '#457b9d',
    'Family':  '#8338ec',
    'Genus':   '#06d6a0',
    'Species': '#adb5bd',
}

node_colors = [color_palette[rank_map[n]] for n in T.nodes()]
print(f'Tree: {T.number_of_nodes()} nodes, {T.number_of_edges()} edges')

### Tree Layout 1 – Hierarchical (Top-Down) using Reingold-Tilford style

In [ ]:
def hierarchy_pos(G, root='Animalia', width=10.0, vert_gap=1.5, vert_loc=0, xcenter=0.5):
    """Compute a top-down hierarchical layout for a directed tree."""
    def _hierarchy_pos(G, root, width, vert_gap, vert_loc, xcenter, pos, parent=None):
        children = list(G.successors(root))
        if parent:
            children = [c for c in children if c != parent]
        if not children:
            pos[root] = (xcenter, vert_loc)
        else:
            dx = width / len(children)
            nextx = xcenter - width / 2 - dx / 2
            for child in children:
                nextx += dx
                pos = _hierarchy_pos(G, child, width=dx, vert_gap=vert_gap,
                                     vert_loc=vert_loc - vert_gap,
                                     xcenter=nextx, pos=pos, parent=root)
            pos[root] = (xcenter, vert_loc)
        return pos
    return _hierarchy_pos(G, root, width, vert_gap, vert_loc, xcenter, {})

pos_hier = hierarchy_pos(T, root='Animalia', width=14, vert_gap=1.4)

fig, ax = plt.subplots(figsize=(20, 12))
fig.patch.set_facecolor('#0d1b2a')
ax.set_facecolor('#0d1b2a')

nx.draw_networkx_edges(T, pos_hier, ax=ax,
                       edge_color='#4a5568', arrows=True,
                       arrowsize=12, width=0.8, alpha=0.7,
                       connectionstyle='arc3,rad=0.0')

nx.draw_networkx_nodes(T, pos_hier, ax=ax,
                       node_color=node_colors, node_size=320, alpha=0.95)

labels = {n: n.split('(')[0].strip() if '(' in n else n for n in T.nodes()}
nx.draw_networkx_labels(T, pos_hier, labels=labels, ax=ax,
                        font_size=6.5, font_color='white', font_weight='bold')

# Legend
patches = [mpatches.Patch(color=c, label=r) for r, c in color_palette.items()]
ax.legend(handles=patches, loc='lower right', fontsize=9,
          facecolor='#1a2a3a', edgecolor='#4a5568', labelcolor='white')

ax.set_title('Animal Kingdom Taxonomy – Hierarchical (Top-Down) Layout',
             color='white', fontsize=15, pad=16, fontweight='bold')
ax.axis('off')
plt.tight_layout()
plt.savefig('tree_layout1_hierarchical.png', dpi=150, bbox_inches='tight',
            facecolor='#0d1b2a')
plt.show()
print('Saved: tree_layout1_hierarchical.png')

### Tree Layout 2 – Radial (Circular) Layout

In [ ]:
def radial_tree_pos(G, root='Animalia'):
    """Convert the hierarchy to polar coordinates for a radial layout."""
    levels = defaultdict(list)
    visited = set()
    queue = [(root, 0)]
    while queue:
        node, depth = queue.pop(0)
        if node in visited:
            continue
        visited.add(node)
        levels[depth].append(node)
        for child in G.successors(node):
            if child not in visited:
                queue.append((child, depth + 1))

    pos = {}
    pos[root] = (0, 0)
    max_depth = max(levels.keys())
    for depth, nodes in levels.items():
        if depth == 0:
            continue
        r = depth / max_depth
        angles = np.linspace(0, 2 * np.pi, len(nodes), endpoint=False)
        for node, angle in zip(nodes, angles):
            pos[node] = (r * np.cos(angle), r * np.sin(angle))
    return pos

pos_radial = radial_tree_pos(T)

fig, ax = plt.subplots(figsize=(16, 16))
fig.patch.set_facecolor('#0d1b2a')
ax.set_facecolor('#0d1b2a')

nx.draw_networkx_edges(T, pos_radial, ax=ax,
                       edge_color='#718096', arrows=False,
                       width=0.9, alpha=0.6)

nx.draw_networkx_nodes(T, pos_radial, ax=ax,
                       node_color=node_colors, node_size=380, alpha=0.95)

nx.draw_networkx_labels(T, pos_radial, labels=labels, ax=ax,
                        font_size=6, font_color='white', font_weight='bold')

# Draw concentric rings for levels
for r in np.linspace(0.15, 1.0, 6):
    circle = plt.Circle((0, 0), r, color='#2d3748', fill=False,
                         linestyle='--', linewidth=0.5, alpha=0.5)
    ax.add_patch(circle)

patches = [mpatches.Patch(color=c, label=r) for r, c in color_palette.items()]
ax.legend(handles=patches, loc='lower right', fontsize=9,
          facecolor='#1a2a3a', edgecolor='#4a5568', labelcolor='white')

ax.set_title('Animal Kingdom Taxonomy – Radial (Circular) Layout',
             color='white', fontsize=15, pad=16, fontweight='bold')
ax.axis('off')
plt.tight_layout()
plt.savefig('tree_layout2_radial.png', dpi=150, bbox_inches='tight',
            facecolor='#0d1b2a')
plt.show()
print('Saved: tree_layout2_radial.png')

---
## Part 2 – Graph Visualization (Les Misérables Character Co-occurrence)

In [ ]:

# Subset of the classic Knuth/Jacomy dataset (top characters by centrality)
characters = [
    'Valjean', 'Javert', 'Fantine', 'Cosette', 'Marius',
    'Thénardier', 'Mme Thénardier', 'Éponine', 'Enjolras',
    'Courfeyrac', 'Combeferre', 'Gavroche', 'Gillenormand',
    'Fauchelevent', 'Bamatabois', 'Bishop Myriel', 'Grantaire',
    'Bossuet', 'Joly', 'Bahorel', 'Pontmercy',
]

weighted_edges = [
    ('Valjean', 'Javert', 10),
    ('Valjean', 'Fantine', 8),
    ('Valjean', 'Cosette', 9),
    ('Valjean', 'Marius', 7),
    ('Valjean', 'Bishop Myriel', 5),
    ('Valjean', 'Thénardier', 6),
    ('Valjean', 'Fauchelevent', 4),
    ('Valjean', 'Gillenormand', 3),
    ('Valjean', 'Enjolras', 4),
    ('Valjean', 'Gavroche', 5),
    ('Valjean', 'Éponine', 4),
    ('Valjean', 'Bamatabois', 3),
    ('Javert', 'Thénardier', 5),
    ('Javert', 'Fantine', 4),
    ('Javert', 'Bamatabois', 3),
    ('Fantine', 'Thénardier', 4),
    ('Fantine', 'Bamatabois', 2),
    ('Cosette', 'Marius', 10),
    ('Cosette', 'Thénardier', 3),
    ('Cosette', 'Mme Thénardier', 3),
    ('Cosette', 'Éponine', 4),
    ('Marius', 'Éponine', 6),
    ('Marius', 'Enjolras', 5),
    ('Marius', 'Courfeyrac', 6),
    ('Marius', 'Combeferre', 4),
    ('Marius', 'Gillenormand', 5),
    ('Marius', 'Pontmercy', 3),
    ('Marius', 'Gavroche', 4),
    ('Thénardier', 'Mme Thénardier', 9),
    ('Thénardier', 'Éponine', 7),
    ('Thénardier', 'Gavroche', 5),
    ('Enjolras', 'Courfeyrac', 7),
    ('Enjolras', 'Combeferre', 7),
    ('Enjolras', 'Grantaire', 6),
    ('Enjolras', 'Bossuet', 5),
    ('Enjolras', 'Joly', 4),
    ('Enjolras', 'Bahorel', 4),
    ('Courfeyrac', 'Combeferre', 6),
    ('Courfeyrac', 'Bossuet', 5),
    ('Courfeyrac', 'Grantaire', 4),
    ('Courfeyrac', 'Gavroche', 4),
    ('Combeferre', 'Bossuet', 5),
    ('Combeferre', 'Joly', 4),
    ('Grantaire', 'Bossuet', 5),
    ('Grantaire', 'Joly', 4),
    ('Bossuet', 'Joly', 6),
    ('Bossuet', 'Bahorel', 4),
    ('Joly', 'Bahorel', 4),
]

G = nx.Graph()
G.add_nodes_from(characters)
for u, v, w in weighted_edges:
    G.add_edge(u, v, weight=w)

print(f'Graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges')

# Node sizes proportional to degree centrality
centrality = nx.degree_centrality(G)
node_sizes_g = [3000 * centrality[n] + 300 for n in G.nodes()]

# Community detection for colouring
from networkx.algorithms.community import greedy_modularity_communities
communities = list(greedy_modularity_communities(G))
community_colors_palette = ['#e63946', '#2a9d8f', '#f4a261', '#8338ec', '#06d6a0']
node_community = {}
for i, comm in enumerate(communities):
    for node in comm:
        node_community[node] = community_colors_palette[i % len(community_colors_palette)]

node_colors_g = [node_community[n] for n in G.nodes()]

### Graph Layout 1 – Force-Directed (Spring) Layout

In [ ]:
pos_spring = nx.spring_layout(G, seed=42, k=2.5, iterations=100)

edge_weights = [G[u][v]['weight'] for u, v in G.edges()]
max_w = max(edge_weights)
edge_widths = [1.0 + 3.0 * (w / max_w) for w in edge_weights]

fig, ax = plt.subplots(figsize=(16, 12))
fig.patch.set_facecolor('#0d1b2a')
ax.set_facecolor('#0d1b2a')

nx.draw_networkx_edges(G, pos_spring, ax=ax,
                       width=edge_widths, edge_color='#4a5568', alpha=0.6)

nx.draw_networkx_nodes(G, pos_spring, ax=ax,
                       node_color=node_colors_g,
                       node_size=node_sizes_g, alpha=0.95)

nx.draw_networkx_labels(G, pos_spring, ax=ax,
                        font_size=8, font_color='white', font_weight='bold')

comm_patches = [
    mpatches.Patch(color=community_colors_palette[i], label=f'Community {i+1}')
    for i in range(len(communities))
]
ax.legend(handles=comm_patches, loc='lower right', fontsize=9,
          facecolor='#1a2a3a', edgecolor='#4a5568', labelcolor='white')

ax.set_title('Les Misérables – Force-Directed (Spring) Layout\n'
             'Node size = degree centrality | Colour = community | Edge width = co-occurrence weight',
             color='white', fontsize=13, pad=14, fontweight='bold')
ax.axis('off')
plt.tight_layout()
plt.savefig('graph_layout1_spring.png', dpi=150, bbox_inches='tight',
            facecolor='#0d1b2a')
plt.show()
print('Saved: graph_layout1_spring.png')

### Graph Layout 2 – Circular Layout

In [ ]:
# Sort nodes by community so same-community nodes cluster together on the circle
sorted_nodes = sorted(G.nodes(), key=lambda n: (list(communities).index(
    next(c for c in communities if n in c)), n))

pos_circ = nx.circular_layout(G, scale=2.0)
# Re-order positions
angles = np.linspace(0, 2 * np.pi, len(sorted_nodes), endpoint=False)
pos_circ = {node: (2.0 * np.cos(a), 2.0 * np.sin(a))
            for node, a in zip(sorted_nodes, angles)}

fig, ax = plt.subplots(figsize=(16, 16))
fig.patch.set_facecolor('#0d1b2a')
ax.set_facecolor('#0d1b2a')

# Draw edges with curvature so they don't overlap in the middle
for u, v, data in G.edges(data=True):
    w = data['weight']
    alpha = 0.25 + 0.6 * (w / max_w)
    lw = 0.5 + 2.5 * (w / max_w)
    ax.annotate('',
                xy=pos_circ[v], xycoords='data',
                xytext=pos_circ[u], textcoords='data',
                arrowprops=dict(arrowstyle='-', color='#718096',
                                lw=lw, alpha=alpha,
                                connectionstyle='arc3,rad=0.25'))

nx.draw_networkx_nodes(G, pos_circ, ax=ax,
                       nodelist=sorted_nodes,
                       node_color=[node_community[n] for n in sorted_nodes],
                       node_size=[3000 * centrality[n] + 300 for n in sorted_nodes],
                       alpha=0.95)

nx.draw_networkx_labels(G, pos_circ, ax=ax,
                        font_size=8, font_color='white', font_weight='bold')

ax.legend(handles=comm_patches, loc='lower right', fontsize=9,
          facecolor='#1a2a3a', edgecolor='#4a5568', labelcolor='white')

ax.set_title('Les Misérables – Circular Layout (nodes ordered by community)\n'
             'Node size = degree centrality | Colour = community | Edge width = co-occurrence weight',
             color='white', fontsize=13, pad=14, fontweight='bold')
ax.axis('off')
plt.tight_layout()
plt.savefig('graph_layout2_circular.png', dpi=150, bbox_inches='tight',
            facecolor='#0d1b2a')
plt.show()
print('Saved: graph_layout2_circular.png')

---
## Bonus – Interactive HTML Visualizations (GitHub Pages ready)

The cells below export self-contained `.html` files that can be hosted directly on GitHub Pages.

In [ ]:

edge_x, edge_y = [], []
for u, v in T.edges():
    x0, y0 = pos_hier[u]
    x1, y1 = pos_hier[v]
    edge_x += [x0, x1, None]
    edge_y += [y0, y1, None]

edge_trace = go.Scatter(x=edge_x, y=edge_y,
                        line=dict(width=0.8, color='#4a5568'),
                        hoverinfo='none', mode='lines')

node_x = [pos_hier[n][0] for n in T.nodes()]
node_y = [pos_hier[n][1] for n in T.nodes()]
node_text = [f"{n}<br>Rank: {rank_map[n]}" for n in T.nodes()]
node_col_hex = [color_palette[rank_map[n]] for n in T.nodes()]

node_trace = go.Scatter(
    x=node_x, y=node_y, mode='markers+text',
    hoverinfo='text', hovertext=node_text,
    text=[labels[n] for n in T.nodes()],
    textposition='top center',
    textfont=dict(size=9, color='white'),
    marker=dict(size=14, color=node_col_hex,
                line=dict(width=1, color='white')))

fig_tree = go.Figure(data=[edge_trace, node_trace],
    layout=go.Layout(
        title=dict(text='Animal Kingdom Taxonomy – Interactive Hierarchical Layout',
                   font=dict(color='white', size=16)),
        paper_bgcolor='#0d1b2a', plot_bgcolor='#0d1b2a',
        showlegend=False,
        xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
        yaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
        height=750,
        margin=dict(l=20, r=20, t=60, b=20)
    ))

fig_tree.write_html('tree_interactive.html', include_plotlyjs='cdn')
print('Saved: tree_interactive.html')

In [ ]:

edge_x_g, edge_y_g = [], []
for u, v in G.edges():
    x0, y0 = pos_spring[u]
    x1, y1 = pos_spring[v]
    edge_x_g += [x0, x1, None]
    edge_y_g += [y0, y1, None]

edge_trace_g = go.Scatter(x=edge_x_g, y=edge_y_g,
                          line=dict(width=0.8, color='#4a5568'),
                          hoverinfo='none', mode='lines')

gnode_x = [pos_spring[n][0] for n in G.nodes()]
gnode_y = [pos_spring[n][1] for n in G.nodes()]
gnode_text = [f"{n}<br>Degree: {G.degree(n)}<br>Centrality: {centrality[n]:.3f}" for n in G.nodes()]
gnode_sizes = [10 + 30 * centrality[n] for n in G.nodes()]

node_trace_g = go.Scatter(
    x=gnode_x, y=gnode_y, mode='markers+text',
    hoverinfo='text', hovertext=gnode_text,
    text=list(G.nodes()),
    textposition='top center',
    textfont=dict(size=9, color='white'),
    marker=dict(size=gnode_sizes, color=node_colors_g,
                line=dict(width=1, color='white')))

fig_graph = go.Figure(data=[edge_trace_g, node_trace_g],
    layout=go.Layout(
        title=dict(text='Les Misérables Co-occurrence – Interactive Force-Directed Layout',
                   font=dict(color='white', size=16)),
        paper_bgcolor='#0d1b2a', plot_bgcolor='#0d1b2a',
        showlegend=False,
        xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
        yaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
        height=750,
        margin=dict(l=20, r=20, t=60, b=20)
    ))

fig_graph.write_html('graph_interactive.html', include_plotlyjs='cdn')
print('Saved: graph_interactive.html')

---
## Summary

| Dataset | Nodes | Edges | Layout 1 | Layout 2 |
|---|---|---|---|---|
| Animal Kingdom Taxonomy | 42 | 41 | Hierarchical (top-down) | Radial (circular) |
| Les Misérables Co-occurrence | 21 | 47 | Force-directed (Spring) | Circular (community-ordered) |

**GitHub Pages:** Host `tree_interactive.html` and `graph_interactive.html` from your repository's `docs/` folder or `gh-pages` branch, then paste the resulting URLs here:

- Tree interactive: `https://eiman-j.github.io/Graph-Tree-Visualization-A4/tree_interactive.html`
- Graph interactive: `https://eiman-j.github.io/Graph-Tree-Visualization-A4/graph_interactive.html`